In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

NOTE: this notebook requires the latest version of aiida-grouppathx.
You can update it with:
```
pip install -U git+https://github.com/zhubonan/aiida-grouppathx
```

In [2]:
from aiida_vasp.workchains.v2 import VaspRelaxUpdater, VaspBuilderUpdater
from aiida_grouppathx import GroupPathX, decorate_with_exit_status
from ase.io import read
from ase.visualize import view
from aiida import orm
from aiida_user_addons.process.transform import make_vac, make_supercell, rattle, get_primitive

In [3]:
from aiida_user_addons.tools.pymatgen import load_mp_struct

In [4]:
basepath = GroupPathX('tlcuse2-defect')
workpath = basepath['workflows']


## Compute Se, Cu

These calculations has been done already - we simply need to import them into the current GroupPath

In [7]:
workpath['se_bulk'] =  GroupPathX('incuse2-defect/workflows/se_bulk').node

In [8]:
workpath['cu_bulk'] =  GroupPathX('incuse2-defect/workflows/cu_bulk').node

## Compute Tl

In [9]:
structure = load_mp_struct('mp-82')
upd = VaspRelaxUpdater().apply_preset(structure, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1}
                                     )
upd.set_resources(tot_num_mpiprocs=32, num_machines=1)
upd.set_options(max_wallclock_seconds=3600, queue_name='xhhctdnormal')
upd.set_label(f'{structure.get_pymatgen().composition.reduced_formula} RELAX')
upd.set_relax_settings(algo='rd')
upd.builder

running = upd.submit()
# Note that this is actually the 8 atom conventional cell
workpath['tl_bulk'] = running

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

/home/bonan/miniconda3/envs/aiida/lib/python3.12/site-packages/aiida_user_addons/tools/pymatgen.py:57: AiidaDeprecationWarning: `StructureData.set_extra` is deprecated, use `StructureData.base.extras.set` instead. (this will be removed in v3)
  strucd.set_extra("mp_id", mp_id)
/home/bonan/miniconda3/envs/aiida/lib/python3.12/site-packages/aiida_user_addons/tools/pymatgen.py:59: AiidaDeprecationWarning: `StructureData.set_extra` is deprecated, use `StructureData.base.extras.set` instead. (this will be removed in v3)
  strucd.set_extra("mp_magmom", magmom)


## Compute TlCuSe2

In [10]:
structure = load_mp_struct('mp-14090')
upd = VaspRelaxUpdater().apply_preset(structure, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1}
                                     )
upd.set_resources(tot_num_mpiprocs=32, num_machines=1)
upd.set_options(max_wallclock_seconds=3600, queue_name='xhhctdnormal')
upd.set_label(f'{structure.get_pymatgen().composition.reduced_formula} RELAX')
upd.set_relax_settings(algo='rd')
upd.builder


running = upd.submit()
# Note that this is actually the 8 atom conventional cell
workpath['tlcuse2_bulk'] = running

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

/home/bonan/miniconda3/envs/aiida/lib/python3.12/site-packages/aiida_user_addons/tools/pymatgen.py:57: AiidaDeprecationWarning: `StructureData.set_extra` is deprecated, use `StructureData.base.extras.set` instead. (this will be removed in v3)
  strucd.set_extra("mp_id", mp_id)
/home/bonan/miniconda3/envs/aiida/lib/python3.12/site-packages/aiida_user_addons/tools/pymatgen.py:59: AiidaDeprecationWarning: `StructureData.set_extra` is deprecated, use `StructureData.base.extras.set` instead. (this will be removed in v3)
  strucd.set_extra("mp_magmom", magmom)


In [20]:
workpath.show_tree(decorate_with_exit_status)

workflows
├── cu_bulk [0]
├── se_bulk [0]
├── tl_bulk [0]
├── tlcuse2_222_V_Cu [waiting]
├── tlcuse2_222_V_Se [waiting]
├── tlcuse2_222_V_Tl [waiting]
├── tlcuse2_222_supercell [waiting]
└── tlcuse2_bulk [waiting]



In [19]:
from ase.visualize import view


In [21]:
view(workpath['tlcuse2_bulk'].node.inputs.structure.get_ase(), viewer='weas')

WeasWidget(children=(BaseWidget(atoms={'species': {'Tl': 'Tl', 'Cu': 'Cu', 'Se': 'Se'}, 'cell': [5.37034214, 0…

## Proceed with defect calculation

The bulk structure is need to generate supercell and defect supercells

In [13]:
ref_structure = workpath['tlcuse2_bulk'].node.outputs.relax.structure
for elem in ['Tl', 'Cu', 'Se']:
    # Find the first occurance of the element
    # This assumes all atoms of the same element are equivalent by symmetry, which may not be the case
    # A better way is to spglib to find unique sites of each element and calculate for them all
    i_elem = ref_structure.get_ase().get_chemical_symbols().index(elem) 
    #  This generate a vacancy cell
    vac_cell = make_vac(ref_structure, [i_elem] , [2,2,2])
    print('Defect cell formula', vac_cell.get_formula())
    upd = VaspRelaxUpdater().apply_preset(vac_cell, 
                                          code='vasp-6.3.2@sugon-xh-v2',
                                          overrides={'ispin': 1,  # I use ISPIN=1 for simplicty, also it may OVERESTIMATE the formation energy
                                                     'ncore':8, 'kpar':4, 'lorbit': None}
                                         )
    upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
    upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
    upd.set_label(f'TlCuSe2 222 V_{elem} RELAX')  # Set the label
    upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
    upd.builder
    running = upd.submit()
    
    workpath.add_node(running, f'tlcuse2_222_V_{elem}')

05/31/2025 10:19:51 PM <2445118> aiida.engine.processes.functions: [INFO] Executing process function, current stack status: 25 frames of 3000
05/31/2025 10:19:52 PM <2445118> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662753>: Broadcasting state change: state_changed.created.running
05/31/2025 10:19:52 PM <2445118> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662753>: Broadcasting state change: state_changed.running.finished


Defect cell formula Cu16Se32Tl15


05/31/2025 10:19:52 PM <2445118> aiida.engine.processes.functions: [INFO] Executing process function, current stack status: 25 frames of 3000
05/31/2025 10:19:52 PM <2445118> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662769>: Broadcasting state change: state_changed.created.running
05/31/2025 10:19:52 PM <2445118> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662769>: Broadcasting state change: state_changed.running.finished


Defect cell formula Cu15Se32Tl16


05/31/2025 10:19:53 PM <2445118> aiida.engine.processes.functions: [INFO] Executing process function, current stack status: 25 frames of 3000
05/31/2025 10:19:53 PM <2445118> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662788>: Broadcasting state change: state_changed.created.running
05/31/2025 10:19:53 PM <2445118> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662788>: Broadcasting state change: state_changed.running.finished


Defect cell formula Cu16Se31Tl16


In [14]:
ref_222 = make_supercell(workpath['tlcuse2_bulk'].node.outputs.relax.structure,[2,2,2])['structure']

upd = VaspRelaxUpdater().apply_preset(ref_222, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1,
                                                 'ncore':8, 'kpar':4, 'lorbit': None,}
                                     )
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
upd.set_label('TlCuSe2 222 SUPERCELL')
upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
upd.builder

running = upd.submit()

workpath.add_node(running, 'tlcuse2_222_supercell')

05/31/2025 10:20:17 PM <2445118> aiida.engine.processes.functions: [INFO] Executing process function, current stack status: 25 frames of 3000
05/31/2025 10:20:17 PM <2445118> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662818>: Broadcasting state change: state_changed.created.running
05/31/2025 10:20:17 PM <2445118> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<662818>: Broadcasting state change: state_changed.running.finished


In [15]:
workpath.show_tree()

workflows
├── cu_bulk *
├── se_bulk *
├── tl_bulk *
├── tlcuse2_222_V_Cu *
├── tlcuse2_222_V_Se *
├── tlcuse2_222_V_Tl *
├── tlcuse2_222_supercell *
└── tlcuse2_bulk *



## Compute the formation energy

In [17]:
def read_energy(path):
    return path.get_node().outputs.misc['total_energies']['energy_extrapolated']
def read_energy_per_atom(path):
    node = path.get_node()
    eng = node.outputs.misc['total_energies']['energy_extrapolated']
    return eng / len(node.inputs.structure.sites)

In [18]:
def show_formation_energy(supercell, v_hg, elemental):
    elem = workpath[elemental].node.outputs.relax.structure.get_formula()
    evac = read_energy(workpath[v_hg])
    print(f'Vacancy bearing cell: {evac:.5f} eV')
    ebulk = read_energy(workpath[supercell])
    print(f'Bulk cell: {ebulk: .5f} eV')
    e_hg = read_energy_per_atom(workpath[elemental])
    print(f'Energy per {elem} atom: {e_hg: .5f} eV')
    e_vac = evac + e_hg - ebulk
    print(f'Vacancy formation energy: {e_vac:.5f} eV')
    return e_vac

In [32]:
forms = {}
for elem in ['Tl',  'Cu', 'Se']:
    forms[elem] = show_formation_energy('tlcuse2_222_supercell', 
                                        f'tlcuse2_222_V_{elem}', f'{elem.lower()}_bulk')

Vacancy bearing cell: -243.39790 eV
Bulk cell: -247.59360 eV
Energy per Tl2 atom: -2.59891 eV
Vacancy formation energy: 1.59680 eV
Vacancy bearing cell: -242.72254 eV
Bulk cell: -247.59360 eV
Energy per Cu atom: -4.29295 eV
Vacancy formation energy: 0.57811 eV
Vacancy bearing cell: -242.55656 eV
Bulk cell: -247.59360 eV
Energy per Se3 atom: -3.85228 eV
Vacancy formation energy: 1.18476 eV


In [35]:
forms

{'Tl': 1.5967955500000244, 'Cu': 0.5781126400000005, 'Se': 1.18475866}

## Check energy and chemical formula using show_tree

In [33]:
def form(path):
    if path.is_node:
        return path.node.inputs.structure.get_formula()
def energy(path):
    if path.is_node:
        if not path.node.is_finished_ok:
            return
        return '{:.4f} eV'.format(path.node.outputs.misc['total_energies']['energy_extrapolated'])

In [34]:
workpath.show_tree(form, energy)

workflows
├── cu_bulk Cu | -4.2929 eV
├── se_bulk Se3 | -11.5568 eV
├── tl_bulk Tl2 | -5.1978 eV
├── tlcuse2_222_V_Cu Cu15Se32Tl16 | -242.7225 eV
├── tlcuse2_222_V_Se Cu16Se31Tl16 | -242.5566 eV
├── tlcuse2_222_V_Tl Cu16Se32Tl15 | -243.3979 eV
├── tlcuse2_222_supercell Cu16Se32Tl16 | -247.5936 eV
└── tlcuse2_bulk Cu2Se4Tl2 | -30.9485 eV

